# EDA — Emerging Market Data

Exploratory data analysis for the emerging-market dataset (Part 2). This notebook examines data quality issues, starting with `construction_type`.

In [0]:
# Construction_type EDA: check for coding inconsistencies across train and test
import pandas as pd
import os

DATA_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data_emerging/'
PROC_DIR = os.path.join(DATA_DIR, 'proc')

# construction_type is a buildings/contents feature — check all sources
sources = {
    'train_buildings (raw)': os.path.join(DATA_DIR, 'policies_buildings.parquet'),
    'train_contents (raw)': os.path.join(DATA_DIR, 'policies_contents.parquet'),
    'test_buildings (raw)': os.path.join(DATA_DIR, 'test_buildings.parquet'),
    'test_contents (raw)': os.path.join(DATA_DIR, 'test_contents.parquet'),
    'proc_buildings': os.path.join(PROC_DIR, 'proc_buildings.parquet'),
    'proc_contents': os.path.join(PROC_DIR, 'proc_contents.parquet'),
}

for label, path in sources.items():
    if not os.path.exists(path):
        print(f'\n{label}: FILE NOT FOUND ({path})')
        continue
    df = pd.read_parquet(path)
    if 'construction_type' not in df.columns:
        print(f'\n{label}: no construction_type column (cols: {df.shape[1]} total)')
        continue
    print(f'\n{"="*60}')
    print(f'{label} — {df.shape[0]:,} rows')
    print(f'{"="*60}')
    vc = df['construction_type'].value_counts(dropna=False)
    print(vc.to_string())
    print(f'  unique values: {df["construction_type"].nunique()} | missing: {df["construction_type"].isna().sum()}')

    # If Year exists, show cross-tab to see the T→W transition
    if 'Year' in df.columns or 'period_start' in df.columns:
        if 'Year' not in df.columns and 'period_start' in df.columns:
            df['_Year'] = pd.to_datetime(df['period_start']).dt.year
            year_col = '_Year'
        else:
            year_col = 'Year'
        print(f'\n  construction_type × {year_col}:')
        ct = pd.crosstab(df[year_col], df['construction_type'], margins=True)
        print(ct.to_string())

In [0]:
# Diagnose why XGBoost D² is negative (avg -0.0961)
# Root cause hypothesis: extreme zero-inflation + small data → systematic under-prediction
import pandas as pd
import numpy as np
import os, pickle

DATA_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data_emerging/'
PROC_DIR = os.path.join(DATA_DIR, 'proc')

# Load processed data
train = {}
for line in ['motor', 'buildings', 'contents']:
    train[line] = pd.read_parquet(os.path.join(PROC_DIR, f'proc_{line}.parquet'))

# Load XGBoost results
with open(os.path.join(PROC_DIR, 'xgboost_v1_results.pkl'), 'rb') as f:
    xgb_results = pickle.load(f)

xgb_oof = xgb_results['oof_predictions']
xgb_cv_scores = xgb_results['cv_scores']

print('='*70)
print('1. ZERO-INFLATION & CLAIM RATES')
print('='*70)
for line in ['motor', 'buildings', 'contents']:
    df = train[line]
    y = df['ultimate_claims'].values
    n = len(y)
    n_zero = (y == 0).sum()
    n_claim = n - n_zero
    print(f'\n{line.upper()}: {n:,} rows | {n_zero:,} zeros ({n_zero/n:.1%}) | {n_claim:,} claims ({n_claim/n:.1%})')
    print(f'  target mean: {y.mean():.2f} | non-zero mean: {y[y>0].mean():.2f} | max: {y.max():.2f}')

print(f'\n{"="*70}')
print(f'2. XGBoost OOF PREDICTIONS vs ACTUALS')
print(f'{"="*70}')
for line in ['motor', 'buildings', 'contents']:
    y = train[line]['ultimate_claims'].values
    pred = xgb_oof[line]
    has_claim = y > 0

    print(f'\n{line.upper()}: XGB CV D2 = {xgb_cv_scores[line]:.4f}')
    print(f'  ALL rows:     pred mean={pred.mean():.1f}  actual mean={y.mean():.1f}  ratio={pred.mean()/y.mean():.2f}')
    print(f'  ZERO claims:   pred mean={pred[~has_claim].mean():.1f}  actual mean={y[~has_claim].mean():.1f}  ({(~has_claim).sum():,} rows)')
    print(f'  NON-ZERO:      pred mean={pred[has_claim].mean():.1f}  actual mean={y[has_claim].mean():.1f}  ratio={pred[has_claim].mean()/y[has_claim].mean():.2f}  ({has_claim.sum():,} rows)')

    # How many claims does XGB predict as near-zero?
    near_zero = (pred[has_claim] < y[has_claim].mean() * 0.1)
    print(f'  Claims predicted <10% of mean: {near_zero.sum()}/{has_claim.sum()} ({near_zero.mean():.1%})')

print(f'\n{"="*70}')
print(f'3. FEATURES-TO-CLAIMS RATIO (dimensionality vs signal)')
print(f'{"="*70}')
for line in ['motor', 'buildings', 'contents']:
    df = train[line]
    n_feats = xgb_results['feature_importances'][line].shape[0]
    n_claims = (df['ultimate_claims'] > 0).sum()
    print(f'  {line}: {n_feats} features, {n_claims:,} claims → {n_feats/n_claims:.2f} features per claim')

print(f'\n{"="*70}')
print(f'4. PARAMETER COUNT vs DATA SIZE')
print(f'{"="*70}')
# XGBoost params: 200 trees × max_depth=4 → up to 16 splits/tree = 3200 leaf nodes
# vs GLM: 20 features + intercept = 21 parameters
xgb_params_effective = 200 * (2**4)  # worst case leaf nodes
for line in ['motor', 'buildings', 'contents']:
    n_claims = (train[line]['ultimate_claims'] > 0).sum()
    print(f'  {line}: XGB ~{xgb_params_effective:,} potential splits vs {n_claims:,} claims (ratio {xgb_params_effective/n_claims:.1f}:1)')
    print(f'         GLM ~21 params vs {n_claims:,} claims (ratio {21/n_claims:.3f}:1)')